# 07. 노선 효과와 위치 효과 분리 — 같은 노선 안에서도 역 근처가 더 줄었나

**문제**: 20분 이내 셀이 많이 줄어든 게 '역 근처라서'인지, 아니면 '그 셀을 지나는 노선(M7111 등)이 원래 많이 줄어서'인지 구분이 안 된다.

**방법**
- 데이터를 **셀 × 노선** 단위로 쪼갠다 (예: 셀 5217의 M7111, 셀 5217의 G7625 …).
- **노선 고정효과**를 넣는다 → 노선 전체의 증감은 제거되고, **같은 노선 안에서 역 근처 정류장과 먼 정류장의 변화율 차이**만 남는다.
- 표준오차는 셀 단위 군집으로 계산한다 (한 셀의 여러 노선은 독립이 아님).
- 2024년 승차가 0인 셀×노선은 변화율을 정의할 수 없어 제외한다.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import norm

data = load_model_data()  # data/processed/model_dataset.csv
X_DUMMY = ["walk_20", "walk_20_60", "ic_access", "competing_share", "pop_20_50", "buildings"]

In [2]:
trips = load_trips()
pairs = (trips.dropna(subset=["셀 ID"])
         .pivot_table(index=["셀 ID", "노선"], columns="연도", values="승차인원", aggfunc="sum", fill_value=0)
         .reset_index().rename(columns={2024: "y2024", 2025: "y2025"}))
pairs["셀 ID"] = pairs["셀 ID"].astype(int)
pairs = pairs.merge(data[["셀 ID", "walk_min"] + X_DUMMY], on="셀 ID")
pairs = pairs[pairs.y2024 > 0].reset_index(drop=True)
print(f"셀×노선 {len(pairs)}쌍 | 셀 {pairs['셀 ID'].nunique()}개 | 노선 {pairs.노선.nunique()}개")
pairs.head()

셀×노선 220쌍 | 셀 132개 | 노선 24개


,셀 ID,노선,y2024,y2025,walk_min,walk_20,walk_20_60,ic_access,competing_share,pop_20_50,buildings
0,44,1200,269,164,51.650000,0.0,1.0,5.071,24.285714,937.0,18.0
1,76,2200,17,35,160.710000,0.0,0.0,1.921,0.000000,33.0,37.0
2,559,3100,82,90,50.225000,0.0,1.0,0.319,0.000000,0.0,46.0
3,619,9710,1,4,166.555000,0.0,0.0,9.029,100.000000,0.0,7.0
4,765,9709,12,14,128.416667,0.0,0.0,4.323,41.666667,139.0,92.0


In [3]:
route_dummies = pd.get_dummies(pairs.노선, prefix="노선", drop_first=True, dtype=float)
models = {
    "노선 FE 없음": pairs[X_DUMMY],
    "노선 FE 포함": pd.concat([pairs[X_DUMMY], route_dummies], axis=1),
}
rows = {}
for name, X in models.items():
    result = poisson_rate(pairs.y2025, pairs.y2024, X, cluster=pairs["셀 ID"])
    rows[name] = coef_table(result, rate=True).loc[[VARIABLE_LABELS[v] for v in X_DUMMY], ["변화율 차이(%)", "p", "유의"]]
pd.concat(rows, axis=1)

노선 FE 없음               노선 FE 포함             
                 변화율 차이(%)       p   유의 변화율 차이(%)       p   유의
보행 20분 이내         -50.4714  0.0001  ***  -60.3744  0.0000  ***
보행 20~60분         -28.8465  0.0151   **  -24.9457  0.0142   **
IC까지 도로거리(km)      -5.1690  0.0275   **   -5.6819  0.0168   **
GTX 경쟁 목적지 비율(%)   -0.4423  0.1172        -0.2711  0.4796     
2024년 20~50대 인구     0.0190  0.0283   **    0.0192  0.1388     
2024년 건축물 수         0.0708  0.7079        -0.0713  0.7415

### 보행시간 연속형 + 노선 고정효과

In [4]:
x_cont = ["walk_min", "ic_access", "competing_share", "pop_20_50", "buildings"]
result = poisson_rate(pairs.y2025, pairs.y2024, pd.concat([pairs[x_cont], route_dummies], axis=1), cluster=pairs["셀 ID"])
print(f"보행시간 10분당 변화율 차이: {(np.exp(result.params['walk_min'] * 10) - 1) * 100:+.1f}% (p={result.pvalues['walk_min']:.4f})")

보행시간 10분당 변화율 차이: +6.3% (p=0.0017)


### 역세권 기준 민감도 (노선 고정효과 포함) — 03의 기준 민감도와 비교

In [5]:
rows = []
for cutoff in (15, 20, 25, 30, 40, 60):
    frame = pairs.assign(near=(pairs.walk_min <= cutoff).astype(float))
    X = pd.concat([frame[["near", "ic_access", "competing_share", "pop_20_50", "buildings"]], route_dummies], axis=1)
    result = poisson_rate(frame.y2025, frame.y2024, X, cluster=frame["셀 ID"])
    rows.append({"기준(분)": cutoff, "기준 이내 셀": frame.loc[frame.near == 1, "셀 ID"].nunique(),
                 "변화율 차이(%)": (np.exp(result.params["near"]) - 1) * 100, "p": result.pvalues["near"]})
pd.DataFrame(rows).round(4)

,기준(분),기준 이내 셀,변화율 차이(%),p
0,15,3,-59.5331,0.0000
1,20,11,-47.5666,0.0000
2,25,16,-29.3227,0.0008
3,30,26,-20.6895,0.0349
4,40,54,-25.0778,0.0014
5,60,85,-26.9439,0.0099


## 결론 ⭐
- **같은 노선 안에서 비교해도 GTX역 가까운 곳이 더 많이 줄었다.** 노선 고정효과를 넣으면 20분 이내 효과가 오히려 커진다: −50% → **−60%**, p < 0.001.
  - 노선별 증감(예: 킨텍스 권역 M7731의 증가)이 역 근처 감소를 일부 가리고 있었다는 뜻이다.
- 연속형 보행시간도 유의하다: 10분당 +6.3%, p = 0.002.
- **03에서 문제였던 기준 민감도가 해결된다**: 노선 고정효과를 넣으면 15·20·25·30·40·60분 **모든 기준에서 유의**하고, 가까울수록 효과가 크다(15분 −60% → 30분 −21%).
  - 03에서 30분 기준이 유의하지 않았던 건 노선 구성 차이가 섞여 있었기 때문이다.
- → **'노선 고정효과 + 셀 군집' 모형을 본모형으로 쓰는 것을 권장한다.**